# Module 4: Practical NLP with Hugging Face Pipelines

In the previous notebook, we explored the inner workings of the Transformer architecture. While understanding the mechanics (like Self-Attention and QKV) is crucial for research, industry practitioners often use high-level abstractions to leverage the power of pre-trained models.

The **Hugging Face `transformers` library** is the definitive toolkit for this. Its most powerful feature is the `pipeline()` API, which simplifies complex NLP tasks into a single line of code.

### How Pipelines Work

A pipeline wraps together three essential steps:
1.  **Tokenization**: Converting raw text into numbers the model understands.
2.  **Model Inference**: Passing those numbers through the Transformer to get predictions.
3.  **Post-processing**: Converting raw predictions back into human-readable labels or text.

```mermaid
graph LR
    Text[Raw Text] --> Tokenizer[Tokenizer]
    Tokenizer --> Model[Pre-trained Model]
    Model --> Post[Post-Processor]
    Post --> Output[Label/Summary/Translation]
    subgraph "The pipeline() Abstraction"
        Tokenizer
        Model
        Post
    end
```

In [1]:
# Install necessary libraries
# 'sentencepiece' is required for many summarization models (like T5/BART)
!pip install transformers[sentencepiece] torch sacremoses -q

from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
import transformers
import pandas as pd
import matplotlib.pyplot as plt

print(f"Transformers version: {transformers.__version__}")

Transformers version: 5.2.0


### Troubleshooting Downloads
If your download gets interrupted, Hugging Face will usually try to resume it. However, if you get a `Pickle` or `EOFError`, it might mean a corrupted file. You can simply **re-run the cell** or restart the kernel. Models are cached in `~/.cache/huggingface` by default.

## 1. Sentiment Analysis

Sentiment analysis determines whether a piece of text is positive, negative, or neutral. With `pipeline("sentiment-analysis")`, you are using a model (usually a fine-tuned DistilBERT) that has seen millions of examples.

In [2]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

data = [
    "This machine learning course is absolutely incredible!",
    "I'm struggling to get this code running, it's very frustrating.",
    "The lecture was okay, but the labs were better."
]

results = classifier(data)

for text, result in zip(data, results):
    print(f"Text: {text}")
    print(f"Label: {result['label']}, Score: {result['score']:.4f}\n")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: This machine learning course is absolutely incredible!
Label: POSITIVE, Score: 0.9999

Text: I'm struggling to get this code running, it's very frustrating.
Label: NEGATIVE, Score: 0.9997

Text: The lecture was okay, but the labs were better.
Label: POSITIVE, Score: 0.9723



## 2. Text Summarization

Summarization models (like BART or T5) condense long documents while preserving the core meaning.

**Note:** To ensure compatibility with all environments, we are using the **explict Model and Tokenizer classes**. This is more robust than the general `pipeline` abstraction when task registries are incomplete.

In [3]:
# Adding redundant imports here just in case the setup cell wasn't re-run!
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_name = "sshleifer/distilbart-cnn-6-6"
print(f"Loading model: {model_name} (this might take a minute)...")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    
    def custom_summarizer(text, max_length=40, min_length=10):
        inputs = tokenizer(text, return_tensors="pt", truncation=True)
        summary_ids = model.generate(inputs["input_ids"], max_length=max_length, min_length=min_length, do_sample=False)
        return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    long_text = """
    The Transformer model, introduced in 2017, marked a paradigm shift in AI. 
    By moving away from recurrent architectures (LSTMs) and embracing the 'Attention' mechanism, 
    it allowed models to process information in parallel rather than sequentially. 
    This scalability led to the birth of Large Language Models (LLMs) like BERT, GPT-3, and Llama. 
    Today, these models are used everywhere, from search engines to creative writing and coding assistants.
    """

    summary_text = custom_summarizer(long_text)
    print(f"\nSummary: {summary_text}")

except Exception as e:
    print(f"\nAn error occurred while loading weights or running inference: {e}")
    print("TIP: If the download was interrupted, try restarting your kernel and running this cell again.")

Loading model: sshleifer/distilbart-cnn-6-6 (this might take a minute)...


Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/262 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/460M [00:00<?, ?B/s]


Summary:  The Transformer model, introduced in 2017, marked a paradigm shift in AI . It allowed models to process information in parallel rather than sequentially . This scalability led to the birth of


d:\setups\ml-dl\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ahmad\.cache\huggingface\hub\models--sshleifer--distilbart-cnn-6-6. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## 3. Zero-Shot Classification

This is a game-changer. You don't need to fine-tune a model for every specific set of labels. You can provide any text and any list of labels, and the model will calculate the probability for each using its pre-trained knowledge.

In [4]:
from transformers import pipeline

# We explicitly use a smaller model ('distilroberta-base') to minimize download time (~300MB)
# The default (BART-Large) is 1.6GB and can be slow to download.
zero_shot = pipeline("zero-shot-classification", model="cross-encoder/nli-distilroberta-base")

text_to_classify = "The new autonomous driving software has shown a 20% improvement in obstacle detection."
candidate_labels = ["technology", "politics", "sports", "health"]

result = zero_shot(text_to_classify, candidate_labels)

print(f"Text: {text_to_classify}")
for label, score in zip(result['labels'], result['scores']):
    print(f"- {label}: {score:.4f}")

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

d:\setups\ml-dl\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ahmad\.cache\huggingface\hub\models--cross-encoder--nli-distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/nli-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Text: The new autonomous driving software has shown a 20% improvement in obstacle detection.
- technology: 0.9376
- sports: 0.0281
- health: 0.0245
- politics: 0.0097


## 4. Bridge to Module 5: LLMs

What you are seeing here is the "Pre-LLM" way of doing things—using specialized models for specific tasks. 

In **Module 5**, we will move to **Generative AI**, where a single model (like GPT or Claude) can do all of the above and more, simply by changing the "Prompt".

| Feature | Pipeline Approach | Generative Approach (LLM) |
| :--- | :--- | :--- |
| **Flexibility** | Task-specific (Sentiment, etc.) | Universal (Any task via prompt) |
| **Efficiency** | Very efficient (small models) | Resource heavy (large models) |
| **Ecosystem** | `transformers` library | APIs (OpenAI/Claude) or local (Ollama) |